In [110]:
# TASK 1

import pandas as pd
import seaborn as sns

df= sns.load_dataset('titanic')
url = "https://raw.githubusercontent.com/agconti/kaggle-titanic/refs/heads/master/data/train.csv"
df_raw = pd.read_csv(url)

df["name"] = df_raw["Name"]
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,name
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,"Braund, Mr. Owen Harris"
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,"Heikkinen, Miss. Laina"
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,"Allen, Mr. William Henry"


In [111]:
print("Shape DataFrame:", df.shape)
df.info()


Shape DataFrame: (891, 16)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
 15  name         891 non-null    str     
dtypes: bool(2), category(2), float64(2), int64(4), str(6)
memory usage: 87.6 KB


In [112]:
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embark_town      2
embarked         2
sex              0
sibsp            0
pclass           0
survived         0
fare             0
parch            0
who              0
class            0
adult_male       0
alive            0
alone            0
name             0
dtype: int64

In [113]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Kolom mana yang punya nilai minimum atau maksimum yang terlihat tidak wajar? : 
1. Kolom age memiliki nilai minimum sekitar 0.42, yang menunjukkan adanya bayi.
2. Kolom fare memiliki nilai maksimum sekitar 512, jauh lebih tinggi dibanding nilai median.

In [114]:
filter_1 = df[
    (df['pclass'].isin([1, 2])) &
    (df['age'] > 30) &
    (df['survived'] == 0)
]

print("Jumlah penumpang:", len(filter_1))

Jumlah penumpang: 94


In [115]:
female = df[
    (df['sex'] == 'female') &
    (df['survived'] == 1) &
    (df['age'] < 18)
]

female[['name', 'age', 'pclass', 'embark_town']]

,name,age,pclass,embark_town
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.00,2,Cherbourg
10,"Sandstrom, Miss. Marguerite Rut",4.00,3,Southampton
22,"McGowan, Miss. Anna ""Annie""",15.00,3,Queenstown
39,"Nicola-Yarred, Miss. Jamila",14.00,3,Cherbourg
43,"Laroche, Miss. Simonne Marie Anne Andree",3.00,2,Cherbourg
58,"West, Miss. Constance Mirium",5.00,2,Southampton
68,"Andersson, Miss. Erna Alexandra",17.00,3,Southampton
84,"Ilett, Miss. Bertha",17.00,2,Southampton
156,"Gilnagh, Miss. Katherine ""Katie""",16.00,3,Queenstown
172,"Johnson, Miss. Eleanor Ileen",1.00,3,Southampton


In [116]:
bins = [0, 12, 18, 60, float('inf')]
labels = ['Anak', 'Remaja', 'Dewasa', 'Lansia']

df['kelompok_usia'] = pd.cut(
    df['age'],
    bins=bins,
    labels=labels,
    right=False
)

df[['age', 'kelompok_usia']].head()

,age,kelompok_usia
0,22.0,Dewasa
1,38.0,Dewasa
2,26.0,Dewasa
3,35.0,Dewasa
4,35.0,Dewasa


In [117]:
survival = (
    df.groupby('pclass')
      .agg(
          jumlah_penumpang=('survived', 'count'),
          jumlah_selamat=('survived', 'sum')
      )
      .reset_index()
)

survival['survival_rate'] = (
    survival['jumlah_selamat'] / survival['jumlah_penumpang'] * 100
).round(2)

survival

,pclass,jumlah_penumpang,jumlah_selamat,survival_rate
0,1,216,136,62.96
1,2,184,87,47.28
2,3,491,119,24.24


In [118]:
agg_data = (
    df.groupby(['pclass', 'sex'])
      .agg(
          rata_usia=('age', 'mean'),
          rata_fare=('fare', 'mean')
      )
      .round(2)
)

agg_data

rata_usia  rata_fare
pclass sex                         
1      female      34.61     106.13
       male        41.28      67.23
2      female      28.72      21.97
       male        30.74      19.74
3      female      21.75      16.12
       male        26.51      12.66

In [119]:
pd.crosstab(df['kelompok_usia'], df['survived'], margins=True)

survived,0,1,All
kelompok_usia,,,
Anak,29,39,68
Remaja,23,22,45
Dewasa,353,222,575
Lansia,19,7,26
All,424,290,714


TASK 4 : Interpretasi 

1. Dari agregasi, perempuan kelas 1 memiliki rata_fare tertinggi, yang mencerminkan akses ke fasilitas premium dibanding kelompok lain.
2. Berdasarkan crosstab, kelompok Anak memiliki proporsi selamat lebih tinggi dibanding tidak selamat, mendukung prioritas evakuasi anak-anak saat tragedi Titanic.